In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -q book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
user_counts = df_ratings['user'].value_counts()
df_ratings = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]

df = df_ratings.merge(df_books, on='isbn')

title_counts = df['title'].value_counts()
df = df[df['title'].isin(title_counts[title_counts >= 100].index)]

df = df.drop_duplicates(subset=['user', 'title'])

book_user_matrix = df.pivot(index='title', columns='user', values='rating').fillna(0)
sparse_matrix = csr_matrix(book_user_matrix.values)

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(sparse_matrix)

In [ ]:
def get_recommends(book = ""):
  book_index = book_user_matrix.index.get_loc(book)
  distances, indices = model.kneighbors(
      book_user_matrix.iloc[book_index].values.reshape(1, -1),
      n_neighbors=6)

  recommended_books = [book, []]
  for i in range(5, 0, -1):
    recommended_books[1].append([
        book_user_matrix.index[indices[0][i]],
        float(distances[0][i])])

  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()